# Online Retail II: three business questions

This notebook reads the generated SQLite tables rather than repeating the cleaning pipeline. The source is the real UCI Online Retail II workbook; synthetic data is reserved for smoke tests and is not used below.

In [1]:
from pathlib import Path
import json
import sqlite3
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
metrics = json.loads((ROOT / 'reports/metrics.json').read_text())
connection = sqlite3.connect(ROOT / 'data/processed/ecommerce.db')
metrics

{'rows': 1036899,
 'orders': 39527,
 'customers': 5852,
 'total_revenue': 20120171.276999995,
 'uk_revenue_share': 0.8570486020520178,
 'top_customer_share': 0.030259267757624077,
 'top_10_customer_share': 0.13888854431345907,
 'source_label': 'REAL - UCI Online Retail II',
 'leading_category': 'Home Decor',
 'leading_category_revenue': 6092219.07,
 'leading_category_share': 0.3027916107734236,
 'other_category_share': 0.12582482783802,
 'peak_month': '2011-11',
 'peak_month_revenue': 1457212.47,
 'peak_month_orders': 2751,
 'discount_signal_months': ['2010-02', '2010-05', '2011-02']}

## Q1. Which product categories generate the most revenue?

**Takeaway:** Home Decor leads at £6.09 million, or 30.3% of cleaned merchandise revenue. The mix is not dominated by one category: Kitchen & Dining and Party & Gift Wrap also contribute materially. The keyword taxonomy leaves 12.6% in `Other`, so I would use this split for direction rather than inventory planning.

In [2]:
pd.read_sql_query(
    'SELECT Category, revenue, revenue_share FROM category_performance ORDER BY revenue DESC',
    connection,
).style.format({'revenue': '£{:,.0f}', 'revenue_share': '{:.1%}'})

,Category,revenue,revenue_share
0,Home Decor,"£6,092,219",30.3%
1,Kitchen & Dining,"£5,448,727",27.1%
2,Other,"£2,531,617",12.6%
3,Party & Gift Wrap,"£2,528,332",12.6%
4,Bags & Accessories,"£1,754,098",8.7%
5,Toys & Stationery,"£1,059,770",5.3%
6,Garden & Outdoor,"£484,142",2.4%
7,Crafts & Sewing,"£114,187",0.6%
8,Bath & Personal Care,"£107,079",0.5%


## Q2. Are sales concentrated by geography or customer?

**Takeaway:** Geography is highly concentrated: the United Kingdom contributes 85.7% of cleaned revenue. Customer concentration is much milder: the largest identified customer contributes 3.0%, and the top 10 contribute 13.9%, which looks more like a long tail than a whale-dependent book of business.

In [3]:
countries = pd.read_sql_query(
    'SELECT Country, revenue, revenue_share FROM country_performance ORDER BY revenue DESC LIMIT 10',
    connection,
)
customers = pd.read_sql_query(
    'SELECT CustomerID, revenue, orders FROM customer_performance ORDER BY revenue DESC LIMIT 10',
    connection,
)
display(countries.style.format({'revenue': '£{:,.0f}', 'revenue_share': '{:.1%}'}))
display(customers.style.format({'revenue': '£{:,.0f}'}))

,Country,revenue,revenue_share
0,United Kingdom,"£17,243,965",85.7%
1,EIRE,"£628,875",3.1%
2,Netherlands,"£549,953",2.7%
3,Germany,"£388,830",1.9%
4,France,"£316,949",1.6%
5,Australia,"£168,485",0.8%
6,Spain,"£98,613",0.5%
7,Switzerland,"£94,270",0.5%
8,Sweden,"£86,353",0.4%
9,Denmark,"£68,560",0.3%


,CustomerID,revenue,orders
0,18102,"£608,822",145
1,14646,"£526,752",145
2,14156,"£303,579",144
3,14911,"£276,601",373
4,17450,"£246,973",51
5,13694,"£196,483",143
6,17511,"£175,604",60
7,16446,"£168,472",2
8,16684,"£147,143",55
9,12415,"£144,033",24


## Q3. How do revenue and orders change over time?

**Takeaway:** November 2011 is the peak month at £1.46 million from 2,751 orders, consistent with holiday-season demand. Orders rose while revenue did not in February 2010, May 2010, and February 2011; I would treat those as price or product-mix signals, not proof of discounting, because the dataset has no promotion field. December 2011 is incomplete and should not be compared with a full holiday month.

In [4]:
monthly = pd.read_sql_query(
    'SELECT month, revenue, orders, average_order_value FROM monthly_performance ORDER BY month',
    connection,
)
monthly['orders_up_revenue_not_up'] = (
    monthly['orders'].diff().gt(0) & monthly['revenue'].diff().le(0)
)
monthly.style.format({'revenue': '£{:,.0f}', 'average_order_value': '£{:,.0f}'})

,month,revenue,orders,average_order_value,orders_up_revenue_not_up
0,2009-12,"£801,319",1671,£480,False
1,2010-01,"£613,959",1053,£583,False
2,2010-02,"£539,692",1189,£454,True
3,2010-03,"£764,351",1647,£464,False
4,2010-04,"£649,071",1436,£452,False
5,2010-05,"£645,729",1484,£435,True
6,2010-06,"£699,344",1612,£434,False
7,2010-07,"£634,962",1507,£421,False
8,2010-08,"£676,176",1402,£482,False
9,2010-09,"£871,873",1789,£487,False


## Interpretation boundary

Revenue here is cleaned gross merchandise value, not profit. Missing customer IDs stay in revenue totals but are excluded from customer rankings. Category assignments are reproducible keyword rules over free-text descriptions, not an official product hierarchy.